<div style="
    text-align: center; 
    background: linear-gradient(135deg, #0062ff 0%, #00d4ff 100%); 
    font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; 
    color: white; 
    padding: 35px 20px; 
    border-radius: 15px; 
    box-shadow: 0 10px 25px rgba(0, 98, 255, 0.3);
    margin-bottom: 25px;">
    <div style="font-size: 35px; font-weight: 800; letter-spacing: 1.5px; text-transform: uppercase; line-height: 1.2;">
        Trực Quan Hóa Dữ Liệu - Lab 03
    </div>
    <div style="font-size: 16px; font-weight: 500; margin-top: 10px; font-style: italic; opacity: 0.9;">
        "Xây dựng mô hình dữ liệu và trực quan hóa bằng Power BI"
    </div>
    <div style="font-size: 18px; font-weight: 600; margin-top: 15px; border-top: 1px solid rgba(255,255,255,0.4); display: inline-block; padding-top: 10px; letter-spacing: 1px;">
        NHÓM 05 - FIT-HCMUS
    </div>
</div>

<div style="text-align: center; font-size: 40px; font-weight: bold;">
  TIỀN XỬ LÝ BỘ DỮ LIỆU TMDB MOVIES
</div>

## 1. Giới thiệu và cấu hình ban đầu

### 1.1. Mục tiêu tiền xử lý

Dataset gốc chứa dữ liệu phim từ TMDB (The Movie Database). Nhóm tập trung nghiên cứu và so sánh các đặc điểm, mức độ phổ biến và đánh giá của các phim đến từ ba quốc gia: **Việt Nam**, **Hàn Quốc**, và **Trung Quốc** trong giai đoạn từ năm 2000 đến năm 2025.

**Phạm vi và yêu cầu tiền xử lý:**
- Lọc dữ liệu theo quốc gia sản xuất (chỉ giữ phim có quốc gia sản xuất chứa `Vietnam`, `South Korea`, hoặc `China`).
- Lọc phim có năm phát hành trong khoảng 2000–2025 dựa trên cột `release_date`.
- Chỉ giữ phim có dữ liệu chấm điểm hợp lệ (số điểm và số lượt đánh giá > 0) và có trạng thái `Released`.
- Đổi tên cột sang tiếng Việt không dấu để phục vụ xây dựng mô hình dữ liệu trong Power BI.
- Tạo các cờ kiểm tra chất lượng dữ liệu để phục vụ lọc/slicer trong báo cáo (không xóa dòng thiếu doanh thu/ngân sách/thời lượng).
- Làm sạch ký tự xuống dòng đặc biệt trước khi export để tránh lỗi hiển thị.
- Tách dữ liệu thành các bảng quan hệ Dim/Fact/Bridge phục vụ Import vào Power BI.

### 1.2. Import thư viện và cấu hình đường dẫn

Nạp các thư viện cần thiết cho việc xử lý dữ liệu, đồng thời cấu hình các đường dẫn lưu trữ tệp dữ liệu đầu vào và các tệp kết quả đầu ra. Đường dẫn được cấu hình động để hoạt động chính xác dù chạy từ thư mục gốc hay thư mục chứa notebook.

In [ ]:
import ast
import json
import os
import re
import pandas as pd

# Tự động điều chỉnh đường dẫn nếu chạy từ thư mục gốc hoặc thư mục 'notebooks'
if os.path.basename(os.getcwd()) == "notebooks":
    RAW_DATA_PATH = "../data/raw/TMDB_movie_dataset.csv"
    OUTPUT_CLEAN_PATH = "../data/processed/tmdb_vn_kr_cn_2000_2025_scored.csv"
    POWERBI_DIR = "../data/processed/powerbi/"
else:
    RAW_DATA_PATH = "data/raw/TMDB_movie_dataset.csv"
    OUTPUT_CLEAN_PATH = "data/processed/tmdb_vn_kr_cn_2000_2025_scored.csv"
    POWERBI_DIR = "data/processed/powerbi/"

# Tạo các thư mục đầu ra nếu chưa tồn tại
os.makedirs(os.path.dirname(OUTPUT_CLEAN_PATH), exist_ok=True)
os.makedirs(POWERBI_DIR, exist_ok=True)

### 1.3. Đọc dữ liệu thô và kiểm tra tổng quan

Đọc tệp dữ liệu gốc, in kích thước dữ liệu (shape) và danh sách cột để xác minh dữ liệu đầu vào.

In [ ]:
# Đọc dữ liệu gốc từ file CSV
df = pd.read_csv(RAW_DATA_PATH)

print(f"Shape ban đầu của dataset: {df.shape}")
print("\nDanh sách các cột trong dữ liệu thô:")
print(df.columns.tolist())

# Kiểm tra các cột bắt buộc cho phạm vi lọc ban đầu
required_columns = [
    "production_countries",
    "release_date",
    "vote_average",
    "vote_count",
]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Thiếu các cột bắt buộc: {missing_columns}")
else:
    print("\nTất cả các cột bắt buộc đều tồn tại.")

display(df.head())

## 2. Lọc dữ liệu theo phạm vi nghiên cứu

### 2.1. Lọc theo quốc gia sản xuất

Lọc theo cột `production_countries`, chỉ giữ lại các phim có quốc gia sản xuất chứa `Vietnam`, `South Korea`, hoặc `China`. Đối với `China`, không bao gồm `Hong Kong`, `Taiwan`, `Macau` làm một phần của nhóm China, trừ khi phim đó đồng thời đồng sản xuất với `China`.

In [ ]:
TARGET_COUNTRIES = ["Vietnam", "South Korea", "China"]
TARGET_COUNTRY_SET = set(TARGET_COUNTRIES)


def _is_missing(value):
    """Kiểm tra giá trị thiếu mà không làm lỗi với list/dict."""
    if value is None:
        return True
    if isinstance(value, float) and pd.isna(value):
        return True
    return False


def _parse_json_like_text(value):
    """Thử chuyển chuỗi JSON/list/dict thành object Python."""
    if not isinstance(value, str):
        return value

    text = value.strip()
    if not text:
        return []

    for parser in (json.loads, ast.literal_eval):
        try:
            return parser(text)
        except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
            continue

    return text


def extract_country_names(value):
    """Trích xuất tên quốc gia từ null, text, list hoặc chuỗi JSON-like."""
    if _is_missing(value):
        return []

    parsed_value = _parse_json_like_text(value)
    country_names = []

    def collect_country_name(item):
        if _is_missing(item):
            return

        if isinstance(item, dict):
            name = item.get("name") or item.get("country") or item.get("english_name")
            if name:
                country_names.append(str(name).strip())
            return

        if isinstance(item, list):
            for child_item in item:
                collect_country_name(child_item)
            return

        if isinstance(item, str):
            text = item.strip()
            if not text:
                return

            parsed_text = _parse_json_like_text(text)
            if parsed_text is not text:
                collect_country_name(parsed_text)
                return

            # Tách chuỗi văn bản thường theo các dấu phân cách phổ biến.
            for name in re.split(r"[,;/|]", text):
                clean_name = name.strip()
                if clean_name:
                    country_names.append(clean_name)

    collect_country_name(parsed_value)

    # Loại trùng nhưng giữ nguyên thứ tự xuất hiện.
    unique_country_names = []
    seen = set()
    for name in country_names:
        normalized_name = name.casefold()
        if normalized_name not in seen:
            unique_country_names.append(name)
            seen.add(normalized_name)

    return unique_country_names


def get_selected_countries(value):
    """Chỉ giữ các quốc gia thuộc phạm vi nghiên cứu (Vietnam, South Korea, China)."""
    country_names = extract_country_names(value)
    selected = [c for c in country_names if c in TARGET_COUNTRY_SET]
    return selected


# Tạo các cột quốc gia để phục vụ lọc và phân tách bảng quan hệ sau này
country_df = df.copy()
country_df["selected_countries"] = country_df["production_countries"].apply(get_selected_countries)
country_df["selected_countries_text"] = country_df["selected_countries"].apply(
    lambda countries: ", ".join(countries)
)
country_df["primary_selected_country"] = country_df["selected_countries"].apply(
    lambda countries: countries[0] if countries else pd.NA
)

# Lọc chỉ giữ phim có ít nhất 1 quốc gia trong danh sách được chọn
filtered_country_df = country_df[country_df["selected_countries"].str.len() > 0].copy()

print(f"Số dòng sau khi lọc quốc gia: {len(filtered_country_df):,}")
print("\nPhân bố theo quốc gia chính (primary):")
print(filtered_country_df["primary_selected_country"].value_counts(dropna=False))
display(filtered_country_df[["title", "production_countries", "selected_countries_text"]].head())

### 2.2. Lọc theo năm phát hành 2000–2025

Chuyển đổi cột ngày phát hành sang kiểu dữ liệu datetime, trích xuất năm phát hành và thực hiện lọc phim được phát hành từ năm 2000 đến năm 2025.

In [ ]:
# Chuyển đổi ngày phát hành sang datetime và trích xuất năm phát hành
filtered_country_df["release_date"] = pd.to_datetime(
    filtered_country_df["release_date"],
    errors="coerce",
)
filtered_country_df["release_year"] = filtered_country_df["release_date"].dt.year

# Đếm số dòng không thể chuyển đổi ngày phát hành
invalid_release_date_count = filtered_country_df["release_date"].isna().sum()
print(f"Số dòng không parse được release_date: {invalid_release_date_count:,}")

# Lọc phim phát hành trong giai đoạn 2000-2025
filtered_year_df = filtered_country_df[
    filtered_country_df["release_year"].between(2000, 2025, inclusive="both")
].copy()
filtered_year_df["release_year"] = filtered_year_df["release_year"].astype("Int64")

print(f"Số dòng sau lọc năm phát hành 2000–2025: {len(filtered_year_df):,}")

### 2.3. Lọc phim có dữ liệu chấm điểm và trạng thái Released

Chỉ giữ lại những bộ phim có số điểm trung bình và số lượt đánh giá hợp lệ (> 0). Đồng thời, nếu tồn tại cột trạng thái, chỉ giữ phim ở trạng thái `Released`. Nếu không tồn tại cột trạng thái, chương trình sẽ in ra cảnh báo và bỏ qua bộ lọc trạng thái.

In [ ]:
# Chuyển dữ liệu đánh giá sang kiểu số
filtered_year_df["vote_average"] = pd.to_numeric(filtered_year_df["vote_average"], errors="coerce")
filtered_year_df["vote_count"] = pd.to_numeric(filtered_year_df["vote_count"], errors="coerce")

# Lọc phim có dữ liệu chấm điểm
filtered_score_df = filtered_year_df[
    (filtered_year_df["vote_average"].notna())
    & (filtered_year_df["vote_count"].notna())
    & (filtered_year_df["vote_average"] > 0)
    & (filtered_year_df["vote_count"] > 0)
].copy()

print(f"Số dòng sau lọc dữ liệu chấm điểm: {len(filtered_score_df):,}")

# Lọc theo trạng thái phát hành Released
if "status" in filtered_score_df.columns:
    print("\nPhân bố trạng thái phim trước khi lọc:")
    print(filtered_score_df["status"].value_counts())

    before_status_filter = len(filtered_score_df)
    filtered_df = filtered_score_df[filtered_score_df["status"] == "Released"].copy()

    print(f"\nSố dòng trước lọc trạng thái: {before_status_filter:,}")
    print(f"Số dòng sau lọc trạng thái Released: {len(filtered_df):,}")
else:
    print("\nCanh bao: Cot 'status' khong ton tai trong dataset. Bo qua buoc loc trang thai.")
    filtered_df = filtered_score_df.copy()

print(f"\nSố dòng cuối cùng sau khi lọc: {len(filtered_df):,}")

## 3. Chuẩn hóa và kiểm tra chất lượng dữ liệu

### 3.1. Đổi tên thuộc tính sang tiếng Việt không dấu

Đổi tên cột của dataset sang tiếng Việt không dấu giúp các thuộc tính trong mô hình dữ liệu của Power BI trở nên đồng bộ, dễ hiểu và dễ thao tác.

In [ ]:
# Từ điển đổi tên cột sang tiếng Việt không dấu
rename_mapping = {
    "id": "MaPhim",
    "title": "TenPhim",
    "vote_average": "DiemDanhGiaTB",
    "vote_count": "SoLuotDanhGia",
    "status": "TrangThai",
    "release_date": "NgayPhatHanh",
    "revenue": "DoanhThu",
    "runtime": "ThoiLuong",
    "adult": "PhimNguoiLon",
    "backdrop_path": "DuongDanAnhNen",
    "budget": "NganSach",
    "homepage": "TrangChu",
    "imdb_id": "MaIMDb",
    "original_language": "NgonNguGoc",
    "original_title": "TenGoc",
    "overview": "TomTatNoiDung",
    "popularity": "DoPhoBien",
    "poster_path": "DuongDanPoster",
    "tagline": "CauGioiThieu",
    "genres": "TheLoai",
    "production_companies": "CongTySanXuat",
    "production_countries": "QuocGiaSanXuat",
    "spoken_languages": "NgonNguDuocNoi",
    "keywords": "TuKhoa",
    "release_year": "NamPhatHanh",
    "selected_countries": "QuocGiaDuocChon",
    "selected_countries_text": "QuocGiaDuocChonText",
    "primary_selected_country": "QuocGiaChinh",
}

# Chỉ đổi tên cột đang tồn tại trong dataset để tránh lỗi
existing_rename_mapping = {
    old_name: new_name
    for old_name, new_name in rename_mapping.items()
    if old_name in filtered_df.columns
}

filtered_df = filtered_df.rename(columns=existing_rename_mapping)

print("Danh sách cột sau khi đổi tên:")
print(filtered_df.columns.tolist())

### 3.2. Xử lý tên phim hiển thị cho dashboard

Tạo thêm cột `TenPhimHienThi` (giữ nguyên tên gốc ban đầu để không dịch tự động) và đánh dấu cờ `CanBoSungTenTiengAnh` nếu tên phim gốc chứa các ký tự đặc trưng của tiếng Trung/Hàn/Nhật (CJK). Điều này giúp người thiết kế dashboard sau này nhận diện được những phim cần bổ sung tên tiếng Anh thủ công để hiển thị đẹp mắt hơn.

In [ ]:
def contains_cjk(text):
    """Kiểm tra xem chuỗi văn bản có chứa ký tự Trung/Hàn/Nhật (CJK) hay không."""
    if not isinstance(text, str):
        return False
    # Regular expression kiểm tra dải Unicode CJK
    return bool(re.search(
        r'[\u4e00-\u9fff\u3400-\u4dbf\uac00-\ud7af\u3040-\u309f\u30a0-\u30ff]',
        text
    ))


# Tạo cột tên phim hiển thị (tạm thời giữ nguyên)
filtered_df["TenPhimHienThi"] = filtered_df["TenPhim"]

# Đánh dấu cờ cần bổ sung tên tiếng Anh
filtered_df["CanBoSungTenTiengAnh"] = filtered_df["TenPhim"].apply(contains_cjk)

print(f"Tổng số phim cần bổ sung tên tiếng Anh (chứa ký tự CJK): {filtered_df['CanBoSungTenTiengAnh'].sum():,}")
if filtered_df["CanBoSungTenTiengAnh"].sum() > 0:
    print("\nSố phim cần bổ sung tên tiếng Anh theo quốc gia chính:")
    print(filtered_df[filtered_df["CanBoSungTenTiengAnh"]].groupby("QuocGiaChinh").size())

### 3.3. Bổ sung các cờ kiểm tra chất lượng dữ liệu

Thêm các thuộc tính cờ kiểm tra chất lượng dữ liệu để thuận tiện cho việc tạo slicer/filter trong Power BI mà **không loại bỏ các dòng bị thiếu thông tin tài chính/thời lượng**:
- `DuSoLuotDanhGia`: Đạt ngưỡng tin cậy đánh giá (`SoLuotDanhGia >= 10`).
- `CoDoanhThu`: `True` nếu `DoanhThu > 0` (các giá trị <= 0 được đưa về NaN).
- `CoNganSach`: `True` nếu `NganSach > 0` (các giá trị <= 0 được đưa về NaN).
- `ThoiLuongBatThuong`: `True` nếu phim có `ThoiLuong > 300` phút.

In [ ]:
# Tạo cờ đủ số lượt đánh giá
filtered_df["DuSoLuotDanhGia"] = filtered_df["SoLuotDanhGia"] >= 10

# Xử lý Doanh Thu
if "DoanhThu" in filtered_df.columns:
    filtered_df["DoanhThu"] = pd.to_numeric(filtered_df["DoanhThu"], errors="coerce")
    filtered_df.loc[filtered_df["DoanhThu"] <= 0, "DoanhThu"] = pd.NA
    filtered_df["CoDoanhThu"] = filtered_df["DoanhThu"].notna()
else:
    filtered_df["CoDoanhThu"] = False

# Xử lý Ngân Sách
if "NganSach" in filtered_df.columns:
    filtered_df["NganSach"] = pd.to_numeric(filtered_df["NganSach"], errors="coerce")
    filtered_df.loc[filtered_df["NganSach"] <= 0, "NganSach"] = pd.NA
    filtered_df["CoNganSach"] = filtered_df["NganSach"].notna()
else:
    filtered_df["CoNganSach"] = False

# Xử lý Thời Lượng
if "ThoiLuong" in filtered_df.columns:
    filtered_df["ThoiLuong"] = pd.to_numeric(filtered_df["ThoiLuong"], errors="coerce")
    filtered_df.loc[filtered_df["ThoiLuong"] <= 0, "ThoiLuong"] = pd.NA
    filtered_df["ThoiLuongBatThuong"] = (
        filtered_df["ThoiLuong"].notna() & (filtered_df["ThoiLuong"] > 300)
    )
else:
    filtered_df["ThoiLuongBatThuong"] = False

print("Thống kê số lượng các cờ chất lượng dữ liệu:")
print(f"  - Phim đủ lượt đánh giá (>= 10): {filtered_df['DuSoLuotDanhGia'].sum():,}")
print(f"  - Phim có doanh thu hợp lệ:      {filtered_df['CoDoanhThu'].sum():,}")
print(f"  - Phim có ngân sách hợp lệ:      {filtered_df['CoNganSach'].sum():,}")
print(f"  - Phim có thời lượng bất thường:  {filtered_df['ThoiLuongBatThuong'].sum():,}")

### 3.4. Làm sạch ký tự xuống dòng đặc biệt

Chuẩn hóa các cột kiểu văn bản (như `TomTatNoiDung`, `CauGioiThieu`) bằng cách loại bỏ các ký tự xuống dòng đặc biệt (`\u2028`, `\u2029`, `\r`, `\n`, `\t`) để tránh việc các phần mềm đọc dữ liệu (như VS Code, Power BI) nhận diện sai dòng hoặc cảnh báo lỗi "Unusual line terminators".

In [ ]:
def clean_text_value(value):
    """Thay thế ký tự xuống dòng lạ bằng khoảng trắng và gom khoảng trắng thừa."""
    if not isinstance(value, str):
        return value

    value = (
        value.replace("\u2028", " ")
        .replace("\u2029", " ")
        .replace("\r", " ")
        .replace("\n", " ")
        .replace("\t", " ")
    )
    # Gom các khoảng trắng liên tiếp thành 1 khoảng trắng và strip đầu cuối
    value = re.sub(r"\s+", " ", value).strip()
    return value


# Áp dụng cho các cột text/object
text_cols = filtered_df.select_dtypes(include=["object"]).columns
print(f"Số cột text cần làm sạch: {len(text_cols)}")
print(f"Danh sách các cột text: {text_cols.tolist()}")

for col in text_cols:
    filtered_df[col] = filtered_df[col].apply(clean_text_value)

print("\nDa hoan tat lam sach cac ky tu xuong dong dac biet.")

### 3.5. Báo cáo kiểm tra chất lượng dữ liệu sau xử lý

Xuất tệp dữ liệu sạch tổng thể `tmdb_vn_kr_cn_2000_2025_scored.csv` và in báo cáo thống kê tóm tắt chất lượng dữ liệu sau các bộ lọc để đảm bảo dữ liệu ổn định và phục vụ lưu trữ file clean tổng.

In [ ]:
# Lưu file clean tổng
filtered_df.to_csv(
    OUTPUT_CLEAN_PATH,
    index=False,
    encoding="utf-8-sig",
    lineterminator="\n",
)

print(f"Da xuat tep du lieu sach tong hop: {OUTPUT_CLEAN_PATH}")
print(f"Kích thước tệp: {filtered_df.shape}")

print("\nThống kê số lượng phim theo quốc gia chính:")
print(filtered_df["QuocGiaChinh"].value_counts(dropna=False))

## 4. Tách dữ liệu thành các bảng quan hệ cho Power BI

### 4.1. Tạo bảng DimPhim

Bảng `DimPhim` chứa các thông tin mô tả chi tiết của từng bộ phim. Khóa chính là `MaPhim` (được đảm bảo là duy nhất và không bị trùng lặp).

In [ ]:
# Các thuộc tính thuộc bảng DimPhim
dim_phim_cols = [
    "MaPhim",
    "TenPhim",
    "TenPhimHienThi",
    "TenGoc",
    "TomTatNoiDung",
    "CauGioiThieu",
    "TrangThai",
    "PhimNguoiLon",
    "NgonNguGoc",
    "TrangChu",
    "MaIMDb",
    "DuongDanPoster",
    "DuongDanAnhNen",
    "CanBoSungTenTiengAnh",
]

# Chỉ lấy những cột tồn tại thực tế
existing_dim_phim_cols = [col for col in dim_phim_cols if col in filtered_df.columns]

# Tạo bảng DimPhim và loại bỏ trùng lặp theo MaPhim
dim_phim = filtered_df[existing_dim_phim_cols].copy()
dim_phim = dim_phim.drop_duplicates(subset=["MaPhim"])

print(f"Kích thước bảng DimPhim: {dim_phim.shape}")
print(f"Số lượng MaPhim duy nhất: {dim_phim['MaPhim'].nunique()}")
display(dim_phim.head())

### 4.2. Tạo bảng FactHieuSuatPhim

Bảng `FactHieuSuatPhim` chứa các chỉ số đo lường hiệu suất của phim như đánh giá, doanh thu, ngân sách, độ phổ biến, thời lượng, và các biến phân loại được tính toán thêm. Yêu cầu `MaPhim` không được chứa giá trị null và không bị trùng lặp.

**Các biến tính toán bổ sung:**
- `LoiNhuan` = `DoanhThu` - `NganSach` (chỉ tính khi cả hai giá trị hợp lệ).
- `TySuatDoanhThuNganSach` = `DoanhThu` / `NganSach` (chỉ tính khi cả hai giá trị hợp lệ).
- `CoThoiLuong`: `True` nếu thời lượng không null.
- Các cột phân nhóm `NhomDiemDanhGia`, `NhomDoPhoBien`, `NhomThoiLuong`, `NhomDoanhThu` được chia theo phân vị (quantile) với cơ chế fallback an toàn nếu gặp lỗi `qcut` do trùng lặp hoặc thiếu dữ liệu.

In [ ]:
# Danh sách các cột thuộc FactHieuSuatPhim ban đầu
fact_cols = [
    "MaPhim",
    "NgayPhatHanh",
    "NamPhatHanh",
    "DiemDanhGiaTB",
    "SoLuotDanhGia",
    "DuSoLuotDanhGia",
    "DoPhoBien",
    "ThoiLuong",
    "ThoiLuongBatThuong",
    "NganSach",
    "CoNganSach",
    "DoanhThu",
    "CoDoanhThu",
]
existing_fact_cols = [col for col in fact_cols if col in filtered_df.columns]

# Khởi tạo bảng fact từ filtered_df
fact_hieu_suat = filtered_df[existing_fact_cols].copy()

# Kiểm tra MaPhim không null và không trùng lặp
fact_hieu_suat = fact_hieu_suat.dropna(subset=["MaPhim"])
fact_hieu_suat = fact_hieu_suat.drop_duplicates(subset=["MaPhim"])

# 1. Tính toán LoiNhuan và TySuatDoanhThuNganSach
fact_hieu_suat["LoiNhuan"] = fact_hieu_suat["DoanhThu"] - fact_hieu_suat["NganSach"]
fact_hieu_suat["TySuatDoanhThuNganSach"] = fact_hieu_suat["DoanhThu"] / fact_hieu_suat["NganSach"]

# 2. Cờ CoThoiLuong
fact_hieu_suat["CoThoiLuong"] = fact_hieu_suat["ThoiLuong"].notna()

# 3. Phân nhóm điểm đánh giá (NhomDiemDanhGia)
try:
    fact_hieu_suat["NhomDiemDanhGia"] = pd.qcut(
        fact_hieu_suat["DiemDanhGiaTB"],
        q=[0, 0.25, 0.75, 1.0],
        labels=["Thap", "Trung binh", "Cao"],
    ).astype(str)
except Exception:
    def fallback_rating(val):
        if pd.isna(val):
            return "Chua xac dinh"
        if val < 5.0:
            return "Thap (< 5.0)"
        elif val < 7.0:
            return "Trung binh (5.0 - 6.9)"
        else:
            return "Cao (>= 7.0)"
    fact_hieu_suat["NhomDiemDanhGia"] = fact_hieu_suat["DiemDanhGiaTB"].apply(fallback_rating)

# 4. Phân nhóm độ phổ biến (NhomDoPhoBien)
try:
    fact_hieu_suat["NhomDoPhoBien"] = pd.qcut(
        fact_hieu_suat["DoPhoBien"],
        q=[0, 0.33, 0.66, 1.0],
        labels=["Thap", "Trung binh", "Cao"],
    ).astype(str)
except Exception:
    def fallback_popularity(val):
        if pd.isna(val):
            return "Chua xac dinh"
        if val < 1.0:
            return "Thap (< 1.0)"
        elif val < 5.0:
            return "Trung binh (1.0 - 4.9)"
        else:
            return "Cao (>= 5.0)"
    fact_hieu_suat["NhomDoPhoBien"] = fact_hieu_suat["DoPhoBien"].apply(fallback_popularity)

# 5. Phân nhóm thời lượng (NhomThoiLuong)
try:
    valid_runtime = fact_hieu_suat["ThoiLuong"].dropna()
    if len(valid_runtime.unique()) >= 3:
        runtime_cats = pd.qcut(valid_runtime, q=[0, 0.33, 0.66, 1.0], labels=["Ngan", "Trung binh", "Dai"])
        fact_hieu_suat["NhomThoiLuong"] = runtime_cats.reindex(fact_hieu_suat.index).astype(str).fillna("Chua xac dinh")
    else:
        raise ValueError("Khong du gia tri duy nhat")
except Exception:
    def fallback_runtime(val):
        if pd.isna(val):
            return "Chua xac dinh"
        if val < 60:
            return "Phim ngan (< 60')"
        elif val < 120:
            return "Phim tieu chuan (60' - 119')"
        else:
            return "Phim dai (>= 120')"
    fact_hieu_suat["NhomThoiLuong"] = fact_hieu_suat["ThoiLuong"].apply(fallback_runtime)

# 6. Phân nhóm doanh thu (NhomDoanhThu)
try:
    valid_rev = fact_hieu_suat["DoanhThu"].dropna()
    if len(valid_rev.unique()) >= 3:
        rev_cats = pd.qcut(valid_rev, q=[0, 0.33, 0.66, 1.0], labels=["Thap", "Trung binh", "Cao"])
        fact_hieu_suat["NhomDoanhThu"] = rev_cats.reindex(fact_hieu_suat.index).astype(str).fillna("Chua xac dinh")
    else:
        raise ValueError("Khong du gia tri duy nhat")
except Exception:
    def fallback_revenue(val):
        if pd.isna(val):
            return "Chua xac dinh"
        if val < 2000000:
            return "Thap (< 2M)"
        elif val < 15000000:
            return "Trung binh (2M - 15M)"
        else:
            return "Cao (>= 15M)"
    fact_hieu_suat["NhomDoanhThu"] = fact_hieu_suat["DoanhThu"].apply(fallback_revenue)

print(f"Kích thước bảng FactHieuSuatPhim: {fact_hieu_suat.shape}")
display(fact_hieu_suat.head())

### 4.3. Tạo bảng DimThoiGian

Bảng `DimThoiGian` được xây dựng từ thuộc tính ngày phát hành (`NgayPhatHanh`) duy nhất, chứa các thuộc tính phân rã thời gian bao gồm: `NgayPhatHanh`, `Nam`, `Thang`, `Quy`, `ThapKy` và `GiaiDoan` (phân loại thành các nhóm giai đoạn: 2000-2009, 2010-2014, 2015-2019, 2020-2025).

In [ ]:
# Trích xuất danh sách ngày phát hành duy nhất và không bị thiếu
dim_thoi_gian = filtered_df[["NgayPhatHanh"]].dropna().drop_duplicates().copy()

# Chuyển đổi ngày phát hành sang datetime để phân rã các thuộc tính thời gian
dim_thoi_gian["NgayPhatHanh_dt"] = pd.to_datetime(dim_thoi_gian["NgayPhatHanh"])
dim_thoi_gian["Nam"] = dim_thoi_gian["NgayPhatHanh_dt"].dt.year.astype(int)
dim_thoi_gian["Thang"] = dim_thoi_gian["NgayPhatHanh_dt"].dt.month.astype(int)
dim_thoi_gian["Quy"] = dim_thoi_gian["NgayPhatHanh_dt"].dt.quarter.astype(int)
dim_thoi_gian["ThapKy"] = (dim_thoi_gian["Nam"] // 10 * 10).astype(str) + "s"


# Hàm phân chia giai đoạn
def get_giai_doan(year):
    if 2000 <= year <= 2009:
        return "2000-2009"
    elif 2010 <= year <= 2014:
        return "2010-2014"
    elif 2015 <= year <= 2019:
        return "2015-2019"
    elif 2020 <= year <= 2025:
        return "2020-2025"
    return "Khac"


dim_thoi_gian["GiaiDoan"] = dim_thoi_gian["Nam"].apply(get_giai_doan)

# Định dạng cột ngày phát hành về chuỗi YYYY-MM-DD để dễ liên kết trong Power BI
dim_thoi_gian["NgayPhatHanh"] = dim_thoi_gian["NgayPhatHanh_dt"].dt.strftime("%Y-%m-%d")

# Loại bỏ cột datetime phụ và giữ lại các cột cần thiết
dim_thoi_gian = dim_thoi_gian[["NgayPhatHanh", "Nam", "Thang", "Quy", "ThapKy", "GiaiDoan"]]

# Format lại ngày phát hành trong fact table về định dạng tương tự
if "NgayPhatHanh" in fact_hieu_suat.columns:
    fact_hieu_suat["NgayPhatHanh"] = pd.to_datetime(fact_hieu_suat["NgayPhatHanh"]).dt.strftime("%Y-%m-%d")

print(f"Kích thước bảng DimThoiGian: {dim_thoi_gian.shape}")
display(dim_thoi_gian.head())

### 4.4. Tạo bảng DimQuocGia và BridgePhimQuocGia

- **DimQuocGia**: Được thiết lập thủ công chứa danh mục 3 quốc gia nghiên cứu: Việt Nam (`VN`), Hàn Quốc (`KR`), và Trung Quốc (`CN`).
- **BridgePhimQuocGia**: Được xây dựng bằng cách phân rã (explode) cột danh sách quốc gia được chọn thật sự (`QuocGiaDuocChon`) của từng bộ phim để hỗ trợ liên kết nhiều-nhiều giữa phim và quốc gia. Một phim thuộc nhiều quốc gia nghiên cứu sẽ được tách thành nhiều dòng tương ứng, và loại bỏ trùng lặp cặp khóa `MaPhim - MaQuocGia`.

In [ ]:
# 1. Tạo bảng DimQuocGia thủ công
dim_quoc_gia = pd.DataFrame([
    {"MaQuocGia": "VN", "TenQuocGia": "Vietnam", "KhuVuc": "DongNamA"},
    {"MaQuocGia": "KR", "TenQuocGia": "South Korea", "KhuVuc": "DongA"},
    {"MaQuocGia": "CN", "TenQuocGia": "China", "KhuVuc": "DongA"}
])

# 2. Tạo bảng BridgePhimQuocGia từ QuocGiaDuocChon
bridge_quoc_gia = filtered_df[["MaPhim", "QuocGiaDuocChon"]].copy()

# Parse lại list nếu QuocGiaDuocChon bị lưu ở dạng chuỗi text biểu diễn list
bridge_quoc_gia["QuocGiaDuocChon"] = bridge_quoc_gia["QuocGiaDuocChon"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Phân rã list thành nhiều dòng
bridge_quoc_gia = bridge_quoc_gia.explode("QuocGiaDuocChon")
bridge_quoc_gia = bridge_quoc_gia.dropna(subset=["QuocGiaDuocChon"])

# Map quốc gia sang mã quốc gia tương ứng
country_map = {
    "Vietnam": "VN",
    "South Korea": "KR",
    "China": "CN"
}
bridge_quoc_gia["MaQuocGia"] = bridge_quoc_gia["QuocGiaDuocChon"].map(country_map)

# Giữ lại cột và loại bỏ duplicate
bridge_quoc_gia = bridge_quoc_gia[["MaPhim", "MaQuocGia"]].dropna().drop_duplicates()

print(f"Kích thước bảng DimQuocGia: {dim_quoc_gia.shape}")
display(dim_quoc_gia)

print(f"\nKích thước bảng BridgePhimQuocGia: {bridge_quoc_gia.shape}")
display(bridge_quoc_gia.head())

### 4.5. Tạo bảng DimTheLoai và BridgePhimTheLoai

- **DimTheLoai**: Được tạo bằng cách thu thập và phân tách toàn bộ thể loại phim từ thuộc tính `TheLoai` trong dữ liệu gốc. Các thể loại được giữ nguyên ngôn ngữ gốc (không dịch). Mã thể loại (`MaTheLoai`) được tự động đánh dạng `G001`, `G002`, `G003`... và loại bỏ trùng lặp thể loại.
- **BridgePhimTheLoai**: Bảng liên kết trung gian chứa các cặp `MaPhim - MaTheLoai` để mô tả các thể loại của từng bộ phim, hỗ trợ mối quan hệ nhiều-nhiều và không bị trùng lặp cặp khóa.

In [ ]:
def parse_genres(value):
    """Phân tách thể loại phim từ dạng list, chuỗi JSON hoặc text thường."""
    if _is_missing(value):
        return []
    parsed_value = _parse_json_like_text(value)
    genres = []

    def collect_genres(item):
        if _is_missing(item):
            return
        if isinstance(item, dict):
            name = item.get("name")
            if name:
                genres.append(str(name).strip())
            return
        if isinstance(item, list):
            for child_item in item:
                collect_genres(child_item)
            return
        if isinstance(item, str):
            text = item.strip()
            if not text:
                return
            parsed_text = _parse_json_like_text(text)
            if parsed_text is not text:
                collect_genres(parsed_text)
                return
            for name in re.split(r"[,;/|]", text):
                clean_name = name.strip()
                if clean_name:
                    genres.append(clean_name)

    collect_genres(parsed_value)
    # Loại trùng lặp và loại trừ các thể loại rỗng
    return list(dict.fromkeys([g for g in genres if g]))


# Thu thập toàn bộ thể loại xuất hiện
all_genres_list = []
for val in filtered_df["TheLoai"].dropna():
    all_genres_list.extend(parse_genres(val))

# Tạo danh mục thể loại duy nhất, sắp xếp chữ cái
unique_genres = sorted(list(set(all_genres_list)))

# Tạo bảng DimTheLoai
dim_the_loai = pd.DataFrame({
    "MaTheLoai": [f"G{i:03d}" for i in range(1, len(unique_genres) + 1)],
    "TenTheLoai": unique_genres
})

# 2. Tạo bảng BridgePhimTheLoai
bridge_the_loai_data = []
genre_to_id = dict(zip(dim_the_loai["TenTheLoai"], dim_the_loai["MaTheLoai"]))

for _, row in filtered_df.iterrows():
    ma_phim = row["MaPhim"]
    movie_genres = parse_genres(row["TheLoai"])
    for g in movie_genres:
        ma_the_loai = genre_to_id.get(g)
        if ma_the_loai:
            bridge_the_loai_data.append({"MaPhim": ma_phim, "MaTheLoai": ma_the_loai})

bridge_the_loai = pd.DataFrame(bridge_the_loai_data)

# Đảm bảo không trùng lặp cặp khóa MaPhim - MaTheLoai và không có giá trị null
bridge_the_loai = bridge_the_loai.dropna().drop_duplicates()

print(f"Kích thước bảng DimTheLoai: {dim_the_loai.shape}")
display(dim_the_loai.head())

print(f"\nKích thước bảng BridgePhimTheLoai: {bridge_the_loai.shape}")
display(bridge_the_loai.head())

## 5. Kiểm tra và xuất dữ liệu đầu ra

### 5.1. Kiểm tra khóa chính, khóa ngoại và duplicate

Kiểm tra tính toàn vẹn dữ liệu trước khi xuất ra các tệp CSV. Code thực hiện kiểm tra kích thước các bảng, số lượng trùng lặp, giá trị thiếu ở khóa chính/khóa ngoại, tính duy nhất của khóa chính, tính hợp lệ của tất cả các mối quan hệ khóa ngoại (Fact/Bridge đến Dim) và in kết quả kiểm tra rõ ràng dạng `PASSED`/`FAILED`. Ngoài ra, in các phân bố dữ liệu bổ sung để phục vụ báo cáo.

In [ ]:
print('=' * 60)
print('KIỂM TRA CHẤT LƯỢNG CÁC BẢNG ĐẦU RA')
print('=' * 60)

# 1. In kích thước của từng bảng
print('\n1. Kích thước (Shape) của các bảng:')
print(f'  - DimPhim:             {dim_phim.shape}')
print(f'  - FactHieuSuatPhim:    {fact_hieu_suat.shape}')
print(f'  - DimThoiGian:         {dim_thoi_gian.shape}')
print(f'  - DimQuocGia:          {dim_quoc_gia.shape}')
print(f'  - BridgePhimQuocGia:   {bridge_quoc_gia.shape}')
print(f'  - DimTheLoai:          {dim_the_loai.shape}')
print(f'  - BridgePhimTheLoai:   {bridge_the_loai.shape}')

# 2. Kiểm tra Null ở khóa chính / khóa ngoại
print('\n2. Số lượng giá trị thiếu (Null) ở các cột khóa:')
print(f'  - DimPhim[MaPhim]:                     {dim_phim["MaPhim"].isna().sum()}')
print(f'  - FactHieuSuatPhim[MaPhim]:            {fact_hieu_suat["MaPhim"].isna().sum()}')
print(f'  - FactHieuSuatPhim[NgayPhatHanh]:      {fact_hieu_suat["NgayPhatHanh"].isna().sum()}')
print(f'  - DimThoiGian[NgayPhatHanh]:           {dim_thoi_gian["NgayPhatHanh"].isna().sum()}')
print(f'  - BridgePhimQuocGia[MaPhim]:           {bridge_quoc_gia["MaPhim"].isna().sum()}')
print(f'  - BridgePhimQuocGia[MaQuocGia]:        {bridge_quoc_gia["MaQuocGia"].isna().sum()}')
print(f'  - BridgePhimTheLoai[MaPhim]:           {bridge_the_loai["MaPhim"].isna().sum()}')
print(f'  - BridgePhimTheLoai[MaTheLoai]:        {bridge_the_loai["MaTheLoai"].isna().sum()}')

# 3. Kiểm tra Duplicate ở từng bảng
print('\n3. Số dòng trùng lặp (Duplicate) trong từng bảng:')
print(f'  - DimPhim:             {dim_phim.duplicated().sum()}')
print(f'  - FactHieuSuatPhim:    {fact_hieu_suat.duplicated().sum()}')
print(f'  - DimThoiGian:         {dim_thoi_gian.duplicated().sum()}')
print(f'  - DimQuocGia:          {dim_quoc_gia.duplicated().sum()}')
print(f'  - BridgePhimQuocGia (cặp khóa): {bridge_quoc_gia.duplicated(subset=["MaPhim", "MaQuocGia"]).sum()}')
print(f'  - BridgePhimTheLoai (cặp khóa): {bridge_the_loai.duplicated(subset=["MaPhim", "MaTheLoai"]).sum()}')

# 4. Kiểm tra tính toàn vẹn và khóa ngoại (Integrity Checks)
print('\n4. Kết quả kiểm tra tính toàn vẹn (PASSED/FAILED):')

# Khóa chính DimPhim[MaPhim] unique
is_dim_phim_pk_unique = dim_phim['MaPhim'].is_unique
print(f'  - DimPhim[MaPhim] unique: {"PASSED" if is_dim_phim_pk_unique else "FAILED"}')

# Khóa ngoại FactHieuSuatPhim[MaPhim] tồn tại trong DimPhim
fact_phim_diff = set(fact_hieu_suat['MaPhim']) - set(dim_phim['MaPhim'])
print(f'  - FactHieuSuatPhim[MaPhim] tồn tại trong DimPhim: {"PASSED" if len(fact_phim_diff) == 0 else "FAILED"}')

# Khóa ngoại BridgePhimQuocGia[MaPhim] tồn tại trong DimPhim
bridge_qg_phim_diff = set(bridge_quoc_gia['MaPhim']) - set(dim_phim['MaPhim'])
print(f'  - BridgePhimQuocGia[MaPhim] tồn tại trong DimPhim: {"PASSED" if len(bridge_qg_phim_diff) == 0 else "FAILED"}')

# Khóa ngoại BridgePhimTheLoai[MaPhim] tồn tại trong DimPhim
bridge_tl_phim_diff = set(bridge_the_loai['MaPhim']) - set(dim_phim['MaPhim'])
print(f'  - BridgePhimTheLoai[MaPhim] tồn tại trong DimPhim: {"PASSED" if len(bridge_tl_phim_diff) == 0 else "FAILED"}')

# Khóa ngoại BridgePhimQuocGia[MaQuocGia] tồn tại trong DimQuocGia
bridge_qg_code_diff = set(bridge_quoc_gia['MaQuocGia']) - set(dim_quoc_gia['MaQuocGia'])
print(f'  - BridgePhimQuocGia[MaQuocGia] tồn tại trong DimQuocGia: {"PASSED" if len(bridge_qg_code_diff) == 0 else "FAILED"}')

# Khóa ngoại BridgePhimTheLoai[MaTheLoai] tồn tại trong DimTheLoai
bridge_tl_code_diff = set(bridge_the_loai['MaTheLoai']) - set(dim_the_loai['MaTheLoai'])
print(f'  - BridgePhimTheLoai[MaTheLoai] tồn tại trong DimTheLoai: {"PASSED" if len(bridge_tl_code_diff) == 0 else "FAILED"}')

# 5. In số phim theo quốc gia và năm phát hành
print('\n5. Số phim theo quốc gia mục tiêu (tính từ BridgePhimQuocGia):')
qg_counts = bridge_quoc_gia.merge(dim_quoc_gia, on='MaQuocGia')['TenQuocGia'].value_counts()
print(qg_counts)

print('\n6. Số phim theo năm phát hành:')
year_counts = fact_hieu_suat['NamPhatHanh'].value_counts().sort_index()
print(year_counts.to_string())

# 6. In top 10 thể loại nhiều phim nhất
print('\n7. Top 10 thể loại nhiều phim nhất:')
tl_counts = bridge_the_loai.merge(dim_the_loai, on='MaTheLoai')['TenTheLoai'].value_counts().head(10)
print(tl_counts)

### 5.2. Xuất các file CSV cho Power BI

Xuất 7 bảng quan hệ đã chuẩn bị sang các tệp CSV riêng biệt nằm trong thư mục `data/processed/powerbi/` phục vụ import trực tiếp vào Power BI. Định dạng ghi tệp đảm bảo các yêu cầu kỹ thuật:
- `encoding="utf-8-sig"` (để hiển thị đúng ký tự Unicode tiếng Việt khi import vào Power BI).
- `index=False` (không ghi cột số thứ tự dòng).
- `lineterminator="\n"` (đảm bảo xuống dòng ổn định).

In [ ]:
# Cấu hình tham số xuất
export_params = {
    "index": False,
    "encoding": "utf-8-sig",
    "lineterminator": "\n",
}

# Xuất 7 file CSV
dim_phim.to_csv(os.path.join(POWERBI_DIR, "DimPhim.csv"), **export_params)
fact_hieu_suat.to_csv(os.path.join(POWERBI_DIR, "FactHieuSuatPhim.csv"), **export_params)
dim_thoi_gian.to_csv(os.path.join(POWERBI_DIR, "DimThoiGian.csv"), **export_params)
dim_quoc_gia.to_csv(os.path.join(POWERBI_DIR, "DimQuocGia.csv"), **export_params)
bridge_quoc_gia.to_csv(os.path.join(POWERBI_DIR, "BridgePhimQuocGia.csv"), **export_params)
dim_the_loai.to_csv(os.path.join(POWERBI_DIR, "DimTheLoai.csv"), **export_params)
bridge_the_loai.to_csv(os.path.join(POWERBI_DIR, "BridgePhimTheLoai.csv"), **export_params)

print(f"Da xuat toan bo 7 tep du lieu quan he vao thu muc: {POWERBI_DIR}")

### 5.3. Mô tả data model dự kiến trong Power BI

Dưới đây là thiết kế mối quan hệ (relationship) giữa các bảng trong mô hình dữ liệu (Star Schema kết hợp Bridge tables) phục vụ dashboard Power BI:

- **DimPhim[MaPhim]** `1 — *` **FactHieuSuatPhim[MaPhim]** (Mối quan hệ 1 - Nhiều để map thông tin mô tả phim với các chỉ số hiệu suất của phim).
- **DimPhim[MaPhim]** `1 — *` **BridgePhimQuocGia[MaPhim]** (Mối quan hệ 1 - Nhiều để phân rã liên kết quốc gia).
- **DimQuocGia[MaQuocGia]** `1 — *` **BridgePhimQuocGia[MaQuocGia]** (Mối quan hệ 1 - Nhiều để map mã quốc gia sang tên đầy đủ và khu vực).
- **DimPhim[MaPhim]** `1 — *` **BridgePhimTheLoai[MaPhim]** (Mối quan hệ 1 - Nhiều để phân rã liên kết thể loại).
- **DimTheLoai[MaTheLoai]** `1 — *` **BridgePhimTheLoai[MaTheLoai]** (Mối quan hệ 1 - Nhiều để map mã thể loại sang tên thể loại gốc).
- **DimThoiGian[NgayPhatHanh]** `1 — *` **FactHieuSuatPhim[NgayPhatHanh]** (Mối quan hệ 1 - Nhiều giúp phân tích hiệu suất phim theo năm, tháng, quý, giai đoạn).

### 5.4. Kết luận tiền xử lý

Quy trình tiền xử lý dữ liệu TMDB Movies A–Z đã hoàn thành tốt đẹp với các kết quả chính:
- **Lọc dữ liệu chính xác**: Chỉ giữ các phim thuộc 3 quốc gia nghiên cứu (Việt Nam, Hàn Quốc, Trung Quốc) phát hành trong giai đoạn 2000–2025, trạng thái đã phát hành (`Released`) và có thông tin chấm điểm hợp lệ.
- **Chuẩn hóa ngôn ngữ**: Toàn bộ tên cột được đưa về tiếng Việt không dấu. Các chuỗi text được làm sạch ký tự xuống dòng đặc biệt để tránh lỗi hiển thị.
- **Giữ toàn vẹn thông tin**: Không xóa dòng thiếu doanh thu/ngân sách/thời lượng, thay vào đó sử dụng cờ chất lượng dữ liệu.
- **Tạo mô hình dữ liệu quan hệ**: Dữ liệu sạch đã được tách thành 7 bảng Dim/Fact/Bridge theo đúng chuẩn và lưu trữ trong thư mục `data/processed/powerbi/`, sẵn sàng để kết nối trực tiếp vào Power BI.